In [241]:
import pandas as pd
import numpy as np
import datetime



In [242]:


df = pd.read_csv('test_data.csv')


print(df.isnull().sum())
print("\n")



id                     0
Birth_Date            13
Weight                 0
Height                 0
Urban_Rural            0
Occupation             0
Insurance_Type         0
Family_History         0
Cancer_Type            0
Stage_at_Diagnosis     0
Diagnosis_Date         0
Symptoms               0
Tumor_Size             0
Surgery_Date           0
Chemotherapy_Drugs     0
Radiation_Sessions     0
Immunotherapy          0
Targeted_Therapy       0
Recurrence_Status      0
Smoking_History        0
Alcohol_Use            0
dtype: int64




In [243]:
def normalize(series, method='minmax'):
    series = pd.to_numeric(series, errors='coerce')
    if method == 'minmax':
        return (series - series.mean()) / (series.max() - series.min())
    elif method == 'zscore':
        return ((series - series.mean()) / series.std())
    elif method == 'max':
        return series / series.max()

def clean_height_column(series):
    return pd.to_numeric(series.str.replace('cm', '', regex=False).str.strip(), errors='coerce')

def preprocess_dates_and_normalize(df, method):
    df['Birth_Date'] = df['Birth_Date'].astype(str).str.strip()
    df = df[~df['Birth_Date'].str.contains(':')].copy()

    df['Birth_Date'] = pd.to_datetime(df['Birth_Date'], errors='coerce')
    df['Diagnosis_Date'] = pd.to_datetime(df['Diagnosis_Date'], errors='coerce')
    df['Surgery_Date'] = pd.to_datetime(df['Surgery_Date'], errors='coerce')
    df['age_at_diagnosis_days'] = (df['Diagnosis_Date'] - df['Birth_Date']).dt.days
    df['XX'] = (df['Surgery_Date'] - df['Birth_Date']).dt.days
    # df = df[~((df['Surgery_Date'].notna()) & (df['Surgery_Date'] < df['Diagnosis_Date']))].copy()
    df['time_to_surgery_days'] = (df['Surgery_Date'] - df['Diagnosis_Date']).dt.days
    df['time_to_surgery_days'] = df['time_to_surgery_days'].fillna(df['time_to_surgery_days'].max())
    df['age1_normalized'] = normalize(df['age_at_diagnosis_days'], method)
    df['age2_normalized'] = normalize(df['time_to_surgery_days'], method)
    df['age3_normalized'] = normalize(df['XX'], method)
    return df

def normalize_of_item(df, method):
    df['Height'] = clean_height_column(df['Height'])
    for col in ['Weight', 'Height', 'Tumor_Size', 'Radiation_Sessions']:
        df[col + '_norm'] = normalize(df[col], method)
    return df

df = preprocess_dates_and_normalize(df, 'zscore')
df = normalize_of_item(df, 'zscore')
df = df.reset_index(drop=True)
df['id'] = df.index + 1


In [244]:

df['Urban_Rural_encoded'] = df['Urban_Rural'].map({'Urban': 0, 'Rural': 1})
###########################
occupation_columns = ['Unemployed', 'Factory Worker', 'Farmer', 'Office Worker', 'Retired']
for occupation in occupation_columns:
    df[f'Occupation_{occupation}'] = (df['Occupation'] == occupation).astype(int)
##############################
insurance_columns = ['UEBMI', 'NRCMS', 'Self-pay', 'URBMI']
for insurance in insurance_columns:
    df[f'Insurance_{insurance}'] = (df['Insurance_Type'] == insurance).astype(int)
###############################
df['family_History_encoded'] = df['Family_History'].map({'No': 0, 'Yes': 1})
############################
cancer_columns = ['Breast', 'Stomach', 'Cervical', 'Lung', 'Esophageal', 'Colorectal', 'Liver']
for cancer in cancer_columns:
    df[f'Cancer_{cancer}'] = (df['Cancer_Type'] == cancer).astype(int)
##############################
Stage_at_Diagnosis_rank = {
    'I':   '1',
    'II':  '2',
    'III': '3',
    'IV':  '4',
}
df['Stage_at_Diagnosis'] = df['Stage_at_Diagnosis'].astype(str).str.strip()
df['stage_at_Diagnosis.rank'] = df['Stage_at_Diagnosis'].map(Stage_at_Diagnosis_rank).fillna('0')
df['stage_at_Diagnosis.rank'] = normalize(df['stage_at_Diagnosis.rank'], 'minmax')
###############################
df['immunotherapy_encoded'] = df['Immunotherapy'].map({'No': 0, 'Yes': 1})
##############################
df['targeted_Therapy_encoded'] = df['Targeted_Therapy'].map({'No': 0, 'Yes': 1})
##############################
df['recurrence_Status_encoded'] = df['Recurrence_Status'].map({'NO': 0, 'Yes': 1})
                                    #   Recurrence_Status
##############################
Smoking_History_rank = {
    'Never':     '1',
    'Former':    '2',
    'Current':   '3',
}
df['Smoking_History'] = df['Smoking_History'].astype(str).str.strip()
df['smoking_History.rank'] = df['Smoking_History'].map(Smoking_History_rank).fillna('0')
df['smoking_History.rank'] = normalize(df['smoking_History.rank'], 'minmax')
############################
Alcohol_Use_rank = {
    'Never':      '1',
    'Occasional': '2',
    'Regular':    '3',
}
df['Alcohol_Use'] = df['Alcohol_Use'].astype(str).str.strip()
df['alcohol_Use.rank'] = df['Alcohol_Use'].map(Alcohol_Use_rank).fillna('0')
df['alcohol_Use.rank'] = normalize(df['alcohol_Use.rank'], 'minmax')
##################################

In [245]:
symptom_list = [
    'Cough',           # 0
    'Weight Loss',     # 1
    'Nausea',          # 2
    'Vomiting',        # 3
    'Blood in Stool',  # 4
    'Fatigue',         # 5
    'Lump',            # 6
    'Pain',            # 7
    'Swelling'         # 8
]
drug_list = [
    'Paclitaxel',
    'Docetaxel',
    'Doxorubicin',
    'Cyclophosphamide',
    'Fluorouracil',
    'Cisplatin',
    'Gemcitabine',
    'Carboplatin',
    'Irinotecan',
    'Oxaliplatin',
    'Leucovorin',
    'Sorafenib',
]
for symptom in symptom_list:
    df[f'Symptom_{symptom}'] = df['Symptoms'].apply(lambda x: 1 if symptom in str(x).split(',') else 0)
for drug in drug_list:
    df[f'Drug_{drug}'] = df['Chemotherapy_Drugs'].apply(lambda x: 1 if drug in str(x).split(',') else 0)

Feature Engineering

In [246]:
non_zero_mask = df['Radiation_Sessions'] != 0
tumor_per_sessions = df.loc[non_zero_mask, 'Tumor_Size'] / df.loc[non_zero_mask, 'Radiation_Sessions']
max_ratio = tumor_per_sessions.max()
df['tumor_per_sessions'] = np.nan
df.loc[non_zero_mask, 'tumor_per_sessions'] = tumor_per_sessions
df['tumor_per_sessions'] = df['tumor_per_sessions'].fillna(max_ratio)
df['tumor_per_sessions_norm'] = normalize(df['tumor_per_sessions'], 'minmax')
# print(df['tumor_per_sessions'].max())

symptom_columns = [col for col in df.columns if col.startswith('Symptom_')]
df['num_symptoms'] = df[symptom_columns].sum(axis=1)
drug_columns = [col for col in df.columns if col.startswith('Drug_')]
df['num_drugs'] = df[drug_columns].sum(axis=1)



non_zero_symptoms_mask = df['num_symptoms'] != 0
drugs_per_symptom = df.loc[non_zero_symptoms_mask, 'num_drugs'] / df.loc[non_zero_symptoms_mask, 'num_symptoms']
max_dps = drugs_per_symptom.max()
df['drugs_per_symptom'] = np.nan
df.loc[non_zero_symptoms_mask, 'drugs_per_symptom'] = drugs_per_symptom
df['drugs_per_symptom'] = df['drugs_per_symptom'].fillna(max_dps)
df['drugs_per_symptom_norm'] = normalize(df['drugs_per_symptom'], 'minmax')



df['BMI'] = df['Weight'] / ((df['Height'] / 100) ** 2)
def bmi_rank(bmi):
    if bmi < 18.5:
        return 0  # Underweight
    elif bmi < 25:
        return 1  # Normal
    elif bmi < 30:
        return 2  # Overweight
    elif bmi < 35:
        return 3 # Obese I
    elif bmi < 40:
        return 4  # Obese II
    else:
        return 5  # Obese III
df['BMI_rank'] = df['BMI'].apply(bmi_rank)
df['BMI_norm'] = normalize(df['BMI'], 'zscore')

# def get_season(month):
#     return ['Winter', 'Spring', 'Summer', 'Fall'][(month % 12) // 3]
# df['birth_season'] = df['Birth_Date'].dt.month.map(get_season)
# season_columns = ['Winter', 'Spring', 'Summer', 'Fall']
# for season in season_columns:
#     df[f'season_{season}'] = (df['birth_season'] == season).astype(int)




In [247]:

df['age_diff'] = (df['age1_normalized'] - df['age2_normalized']).abs()
df['age_diff'] = normalize(df['age_diff'], 'minmax')
df['tumor_bmi_ratio'] = df['Tumor_Size_norm'] / (df['BMI_norm'] + 0.001)
df['tumor_bmi_ratio'] = normalize(df['tumor_bmi_ratio'], 'minmax')
df['urban_treatment_access'] = df['Urban_Rural_encoded'] * df['Radiation_Sessions_norm']
df['drugs_sessions_ratio'] = df['num_drugs'] / (df['Radiation_Sessions_norm'] + 1)
df['symptom_severity'] = df['num_symptoms'] + (1.5 * df['recurrence_Status_encoded'])
df['insurance_coverage_level'] = (
    2.0 * df['Insurance_UEBMI'] + 
    1.5 * df['Insurance_URBMI'] + 
    1.0 * df['Insurance_NRCMS'] - 
    2.0 * df['Insurance_Self-pay']
)


df['BMI_stage'] = df['BMI'] * df['stage_at_Diagnosis.rank']
df['BMI_stage'] = normalize(df['age_diff'], 'minmax')


df['age_at_diagnosis_years'] = df['age_at_diagnosis_days'] / 365.25
df['age_group'] = pd.cut(df['age_at_diagnosis_years'], bins=[0, 30, 45, 60, 75, 100], labels=['0','1','2','3','4'])
df['age_group'] = normalize(df['age_group'], 'minmax')
# df['Radiation_Sessions_norm'] = normalize(df['Radiation_Sessions'], 'zscore')


from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Cancer_Type_Ranked'] = le.fit_transform(df['Cancer_Type'])
df['Cancer_Type_Ranked'] = normalize(df['Cancer_Type_Ranked'], 'zscore')

columns_to_save = [
    'Cancer_Type_Ranked',
    'drugs_sessions_ratio',
    'urban_treatment_access',
    'age_diff',
    'BMI_stage',



    'age1_normalized', 
    'age2_normalized', 
    'Weight_norm', 'Height_norm', 
    'Tumor_Size_norm', 'Radiation_Sessions_norm',
    'Urban_Rural_encoded', 
    'family_History_encoded',
    'stage_at_Diagnosis.rank',
    'immunotherapy_encoded', 'targeted_Therapy_encoded',
    'recurrence_Status_encoded','smoking_History.rank',
    'alcohol_Use.rank',
    'BMI_norm',
    'tumor_per_sessions_norm',
    'num_drugs', 'num_symptoms',
    'drugs_per_symptom_norm',
    'label',
]

occupation_columns_with_prefix = [f'Occupation_{occupation}' for occupation in occupation_columns]
columns_to_save.extend(occupation_columns_with_prefix)
####################################
insurance_columns_with_prefix = [f'Insurance_{insurance}' for insurance in insurance_columns]
columns_to_save.extend(insurance_columns_with_prefix)
#########################################
cancer_columns_with_prefix = [f'Cancer_{cancer}' for cancer in cancer_columns]
columns_to_save.extend(cancer_columns_with_prefix)
############################################
symptom_columns_with_prefix = [f'Symptom_{symptom}' for symptom in symptom_list]
drug_columns_with_prefix = [f'Drug_{drug}' for drug in drug_list]
columns_to_save.extend(symptom_columns_with_prefix)
columns_to_save.extend(drug_columns_with_prefix)
###################################################
smoke = ['Never', 'Former', 'Current']
for s in smoke:
    df[f'Smoke_{s}'] = (df['Smoking_History'] == s).astype(int)

smoke_columns_with_prefix = [f'Smoke_{s}' for s in smoke]
columns_to_save.extend(smoke_columns_with_prefix)


smokee = ['Never', 'Occasional', 'Regular']
for s in smokee:
    df[f'Smokee_{s}'] = (df['Alcohol_Use'] == s).astype(int)

smoke_columns_with_prefixx = [f'Smokee_{s}' for s in smokee]
columns_to_save.extend(smoke_columns_with_prefixx)
##################################################
existing_columns = [col for col in columns_to_save if col in df.columns]
df[existing_columns].to_csv('pre_processing2.csv', index=False)

Modeling

In [249]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
new_df = pd.read_csv('pre_processing2.csv')

In [251]:
# X = new_df.drop(columns=['label'])
# y = new_df['label']


# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model = LogisticRegression(penalty='l2', C=1000, solver='liblinear')
# model.fit(X_train, y_train)

# y_pred = model.predict(X_test)
# acc = accuracy_score(y_test, y_pred)
# print(f"Accuracy: {acc:.4f}")

# print(y_pred[:20])
# print(y_train[:20])


In [ ]:
# import pandas as pd
# import numpy as np
from catboost import CatBoostClassifier, Pool, cv
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_curve

df = pd.read_csv('pre_processing.csv')
X = df.drop(columns=['label']).replace([np.inf, -np.inf], np.nan).fillna(0)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

train_pool = Pool(X_train, y_train)

params = {
    'iterations': 1000,
    'learning_rate': 0.03,
    'depth': 8,
    'l2_leaf_reg': 3,
    'border_count': 128,
    'random_strength': 1,
    'bagging_temperature': 0.2,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'verbose': 100,
    'early_stopping_rounds': 50,
    'random_seed': 42
}

cv_results = cv(
    pool=train_pool,
    params=params,
    fold_count=5,
    shuffle=True,
    partition_random_seed=42,
    verbose=100,
    plot=False
)

best_iter = len(cv_results)
print(f"Best iteration from CV: {best_iter}")

final_model = CatBoostClassifier(
    iterations=best_iter,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=0
)
final_model.fit(X_train, y_train)

y_proba = final_model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
best_threshold = thresholds[np.argmax(tpr - fpr)]
y_pred_opt = (y_proba >= best_threshold).astype(int)

acc = accuracy_score(y_test, y_pred_opt)
print(f"CatBoost Accuracy (optimized): {acc:.4f}")
print(f"Best threshold: {best_threshold:.4f}")

output_df = pd.DataFrame({
    'index': X_test.index,
    'predicted_label': y_pred_opt,
    'real': y_test.values
})
output_df.to_csv('predictions_catboost_optimized.csv', index=False)





# import pandas as pd
# from catboost import CatBoostClassifier, Pool, cv
# import numpy as np

# df_train = pd.read_csv('pre_processing.csv')
# X_train = df_train.drop(columns=['label']).replace([np.inf, -np.inf], np.nan).fillna(0)
# y_train = df_train['label']

# train_pool = Pool(X_train, y_train)

# params = {
#     'iterations': 1000,
#     'learning_rate': 0.03,
#     'depth': 8,
#     'l2_leaf_reg': 3,
#     'border_count': 128,
#     'random_strength': 1,
#     'bagging_temperature': 0.2,
#     'loss_function': 'Logloss',
#     'eval_metric': 'AUC',
#     'verbose': 100,
#     'early_stopping_rounds': 50,
#     'random_seed': 42
# }

# cv_results = cv(
#     pool=train_pool,
#     params=params,
#     fold_count=5,  # 5-fold cross-validation
#     shuffle=True,
#     partition_random_seed=42,
#     verbose=100,
#     plot=True
# )

# best_iter = cv_results['iterations'].max()
# print(f"Best iteration from CV: {best_iter}")

# final_model = CatBoostClassifier(
#     iterations=best_iter,
#     learning_rate=0.03,
#     depth=8,
#     l2_leaf_reg=3,
#     border_count=128,
#     random_strength=1,
#     bagging_temperature=0.2,
#     loss_function='Logloss',
#     eval_metric='AUC',
#     random_seed=42,
#     verbose=0
# )

# final_model.fit(X_train, y_train)

# df_test = pd.read_csv('pre_processing2.csv')
# X_test = df_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# y_pred = final_model.predict(X_test)

# submission_df = pd.DataFrame({
#     'id': range(1, len(X_test) + 1),  # Assuming the 'id' starts from 1
#     'label': y_pred
# })

# submission_df.to_csv('CCC2.csv', index=False)


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Training on fold [0/5]
0:	test: 0.8351709	best: 0.8351709 (0)	total: 105ms	remaining: 31.5s
100:	test: 0.9249489	best: 0.9249489 (100)	total: 8.29s	remaining: 16.3s
200:	test: 0.9443709	best: 0.9443709 (200)	total: 15.8s	remaining: 7.77s
299:	test: 0.9577546	best: 0.9577546 (299)	total: 23.4s	remaining: 0us

bestTest = 0.9577546117
bestIteration = 299

Training on fold [1/5]
0:	test: 0.8346538	best: 0.8346538 (0)	total: 80.8ms	remaining: 24.2s
100:	test: 0.9261632	best: 0.9261632 (100)	total: 8s	remaining: 15.8s
200:	test: 0.9445131	best: 0.9445131 (200)	total: 15.8s	remaining: 7.77s
299:	test: 0.9583812	best: 0.9583812 (299)	total: 23.3s	remaining: 0us

bestTest = 0.9583811961
bestIteration = 299

Training on fold [2/5]
0:	test: 0.8329841	best: 0.8329841 (0)	total: 96.1ms	remaining: 28.7s
100:	test: 0.9253323	best: 0.9253323 (100)	total: 8.14s	remaining: 16s
200:	test: 0.9441529	best: 0.9441529 (200)	total: 16s	remaining: 7.9s
299:	test: 0.9582717	best: 0.9582717 (299)	total: 23.9s	re

In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_curve

# df = pd.read_csv('pre_processing.csv')
# X = df.drop(columns=['label']).replace([np.inf, -np.inf], np.nan).fillna(0)
# y = df['label']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# lgb_model = LGBMClassifier(
#     n_estimators=300,
#     max_depth=15,
#     learning_rate=0.05,
#     num_leaves=64,
#     class_weight='balanced',
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42,
#     n_jobs=-1
# )
# lgb_model.fit(X_train, y_train)

# y_proba = lgb_model.predict_proba(X_test)[:, 1]
# fpr, tpr, thresholds = roc_curve(y_test, y_proba)
# best_threshold = thresholds[np.argmax(tpr - fpr)]
# y_pred_opt = (y_proba >= best_threshold).astype(int)
# acc = accuracy_score(y_test, y_pred_opt)

# print(f" LightGBM Accuracy (optimized): {acc:.4f}")
# print(f"Best threshold: {best_threshold:.4f}")

# output_df = pd.DataFrame({
#     'index': X_test.index,
#     'predicted_label': y_pred_opt,
#     'real': y_test.values
# })
# output_df.to_csv('lgb_predictions.csv', index=False)

# X = new_df.copy()
# df2 = pd.read_csv('pre_processing.csv')
# XX = df2.drop(columns=['label'])
# YY = df2['label']
# lgb_model = LGBMClassifier(
#     n_estimators=300,
#     max_depth=15,
#     learning_rate=0.05,
#     num_leaves=64,
#     class_weight='balanced',
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42,
#     n_jobs=-1
# )
# lgb_model.fit(XX, YY)
# y_pred = lgb_model.predict(X)
# submission_df = pd.DataFrame({
#     'id': range(1, len(X) + 1),
#     'label': y_pred
# })
# submission_df.to_csv('CCC2.csv', index=False)




In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score, roc_curve
# import pandas as pd
# import numpy as np

# df = pd.read_csv('pre_processing.csv')
# X = df.drop(columns=['label']).replace([np.inf, -np.inf], np.nan).fillna(0)
# y = df['label']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# rf_model = RandomForestClassifier(
#     n_estimators=300,
#     max_depth=20,
#     min_samples_split=4,
#     min_samples_leaf=1,
#     class_weight='balanced',
#     random_state=42,
#     n_jobs=-1
# )
# rf_model.fit(X_train, y_train)

# y_proba = rf_model.predict_proba(X_test)[:, 1]
# fpr, tpr, thresholds = roc_curve(y_test, y_proba)
# best_threshold = thresholds[np.argmax(tpr - fpr)]
# y_pred_opt = (y_proba >= best_threshold).astype(int)
# acc = accuracy_score(y_test, y_pred_opt)
# print(f"Random Forest Accuracy (optimized): {acc:.4f}")
# print(f"Best threshold: {best_threshold:.4f}")

# output_df = pd.DataFrame({
#     'index': X_test.index,
#     'predicted_label': y_pred_opt,
#     'real': y_test.values
# })
# output_df.to_csv('predictions_rf_optimized.csv', index=False)


# import pandas as pd
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score
# from sklearn.model_selection import train_test_split
# import numpy as np
# df = pd.read_csv('pre_processing.csv')
# X = df.drop(columns=['label']).replace([np.inf, -np.inf], np.nan).fillna(0)
# y = df['label']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
# rf_model = RandomForestClassifier(n_estimators=110, random_state=42)
# # rf_model = RandomForestClassifier(
# #     n_estimators=110,
# #     max_depth=15,
# #     min_samples_split=5,
# #     min_samples_leaf=2,
# #     class_weight='balanced',
# #     random_state=42
# # )
# rf_model.fit(X_train, y_train)
# y_pred = rf_model.predict(X_test)
# acc = accuracy_score(y_test, y_pred)
# print(f"Random Forest Accuracy: {acc:.4f}")


# output_df = pd.DataFrame({
#     'index': X_test.index,
#     'predicted_label': y_pred,
#     'real' : y_test.values
# })
# output_df.to_csv('predictions.csv', index=False)

    

# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score
# from sklearn.model_selection import train_test_split
# X = new_df.copy()
# df2 = pd.read_csv('pre_processing.csv')
# XX = df2.drop(columns=['label'])
# YY = df2['label']
# rf_model = RandomForestClassifier(
#     n_estimators=300,
#     max_depth=20,
#     min_samples_split=4,
#     min_samples_leaf=1,
#     class_weight='balanced',
#     random_state=42,
#     n_jobs=-1
# )
# rf_model.fit(XX, YY)
# y_pred = rf_model.predict(X)
# submission_df = pd.DataFrame({
#     'id': range(1, len(X) + 1),
#     'label': y_pred
# })
# submission_df.to_csv('CCC.csv', index=False)


In [255]:
new_df.columns

Index(['Cancer_Type_Ranked', 'drugs_sessions_ratio', 'urban_treatment_access',
       'age_diff', 'BMI_stage', 'age1_normalized', 'age2_normalized',
       'Weight_norm', 'Height_norm', 'Tumor_Size_norm',
       'Radiation_Sessions_norm', 'Urban_Rural_encoded',
       'family_History_encoded', 'stage_at_Diagnosis.rank',
       'immunotherapy_encoded', 'targeted_Therapy_encoded',
       'recurrence_Status_encoded', 'smoking_History.rank', 'alcohol_Use.rank',
       'BMI_norm', 'tumor_per_sessions_norm', 'num_drugs', 'num_symptoms',
       'drugs_per_symptom_norm', 'Occupation_Unemployed',
       'Occupation_Factory Worker', 'Occupation_Farmer',
       'Occupation_Office Worker', 'Occupation_Retired', 'Insurance_UEBMI',
       'Insurance_NRCMS', 'Insurance_Self-pay', 'Insurance_URBMI',
       'Cancer_Breast', 'Cancer_Stomach', 'Cancer_Cervical', 'Cancer_Lung',
       'Cancer_Esophageal', 'Cancer_Colorectal', 'Cancer_Liver',
       'Symptom_Cough', 'Symptom_Weight Loss', 'Symptom_Nausea',

In [ ]:
# import pandas as pd
# import numpy as np
# from xgboost import XGBClassifier
# from sklearn.metrics import accuracy_score
# from sklearn.model_selection import train_test_split
# df = pd.read_csv('pre_processing.csv')
# X = df.drop(columns=['label']).replace([np.inf, -np.inf], np.nan).fillna(0)
# y = df['label']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# xgb_model = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='error', random_state=42, learning_rate=0.05, n_jobs=-1, 
#                           objective='binary:logistic', booster='gbtree', min_child_weight=1, reg_lambda=3, reg_alpha=0.1, max_depth=6)
# xgb_model.fit(X_train, y_train)
# y_pred = xgb_model.predict_proba(X_test)[:, 1]
# y_pred = (y_pred > 0.5).astype(int)
# acc = accuracy_score(y_test, y_pred)
# print(f"XGBoost Accuracy: {acc:.4f}")

# output_df = pd.DataFrame({
#     'index': X_test.index,
#     'predicted_label': y_pred,
#     'real': y_test.values
# })
# output_df.to_csv('xgboost_predictions.csv', index=False)


# import pandas as pd
# import numpy as np
# from xgboost import XGBClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score

# df = pd.read_csv('pre_processing.csv')
# X = df.drop(columns=['label']).replace([np.inf, -np.inf], np.nan).fillna(0)
# y = df['label']

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# xgb_model = XGBClassifier(
#     n_estimators=300,
#     max_depth=6,
#     learning_rate=0.05,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     min_child_weight=3,
#     reg_alpha=0.2,
#     reg_lambda=1.0,
#     use_label_encoder=False,
#     eval_metric='logloss',
#     random_state=42,
#     n_jobs=-1
# )

# xgb_model.fit(X_train, y_train)

# y_pred = xgb_model.predict(X_test)

# acc = accuracy_score(y_test, y_pred)
# print(f"XGBoost Accuracy: {acc:.4f}")

# pd.DataFrame({
#     'index': X_test.index,
#     'predicted_label': y_pred,
#     'real': y_test.values
# }).to_csv('xgboost_predictions.csv', index=False)






# import pandas as pd
# import numpy as np
# from xgboost import XGBClassifier

# df_train = pd.read_csv('pre_processing.csv')
# X = df_train.drop(columns=['label']).replace([np.inf, -np.inf], np.nan).fillna(0)
# y = df_train['label']


# X_submission = pd.read_csv('pre_processing2.csv')

# for col in X_submission.select_dtypes(include='object').columns:
#     X_submission[col] = X_submission[col].astype('category').cat.codes

# X_submission = X_submission.replace([np.inf, -np.inf], np.nan).fillna(0)

# X_submission = X_submission[X.columns]

# model = XGBClassifier(
#     n_estimators=300,
#     max_depth=6,
#     learning_rate=0.05,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     min_child_weight=3,
#     reg_alpha=0.2,
#     reg_lambda=1.0,
#     use_label_encoder=False,
#     eval_metric='logloss',
#     random_state=42,
#     n_jobs=-1
# )
# model.fit(X, y)

# y_sub_pred = model.predict(X_submission)

# submission_df = pd.DataFrame({
#     'id': range(1, len(X_submission) + 1),
#     'label': y_sub_pred
# })
# submission_df.to_csv('submission_xgb.csv', index=False)









# from xgboost import XGBClassifier
# import pandas as pd

# X = new_df.copy()
# df2 = pd.read_csv('pre_processing.csv')
# XX = df2.drop(columns=['label'])
# YY = df2['label']

# xgb_model = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='error', random_state=42, learning_rate=0.05, n_jobs=-1, 
#                           objective='binary:logistic', booster='gbtree', min_child_weight=1, reg_lambda=3, reg_alpha=0.1, max_depth=12)
# xgb_model.fit(XX, YY)

# y_pred = xgb_model.predict(X)

# submission_df = pd.DataFrame({
#     'id': range(1, len(X) + 1),
#     'label': y_pred
# })
# submission_df.to_csv('CCC_xgb.csv', index=False)


In [ ]:
# y_proba_1 = xgb_model.predict_proba(X)[:, 1]
# y_proba_2 = rf_model.predict_proba(X)[:, 1]

# y_pred = (((y_proba_1 + y_proba_2)/2) > 0.50).astype(int)
# acc = accuracy_score(y, y_pred)
# print(f"XGBoost and RF Accuracy: {acc:.4f}")

# output_df = pd.DataFrame({
#     'index': X.index,
#     'predicted_label': y_pred,
#     'real': y.values
# })
# output_df.to_csv('comb_predictions.csv', index=False)

# df2 = pd.read_csv('pre_processing.csv')
# XX = df2.drop(columns=['label'])
# YY = df2['label']



# import pandas as pd
# import numpy as np
# from xgboost import XGBClassifier
# from sklearn.ensemble import RandomForestClassifier

# df = pd.read_csv('pre_processing.csv')
# X_train = df.drop(columns=['label']).replace([np.inf, -np.inf], np.nan).fillna(0)
# y_train = df['label']

# X_test = new_df.replace([np.inf, -np.inf], np.nan).fillna(0)


# xgb_model.fit(X_train, y_train)
# rf_model.fit(X_train, y_train)

# y_proba_1 = xgb_model.predict_proba(X_test)[:, 1]
# y_proba_2 = rf_model.predict_proba(X_test)[:, 1]
# y_pred = (((y_proba_1 + y_proba_2) / 2) > 0.5).astype(int)

# submission_df = pd.DataFrame({
#     'id': range(1, len(X_test) + 1),
#     'label': y_pred
# })
# submission_df.to_csv('CCC_comb.csv', index=False)



In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.ensemble import RandomForestClassifier, VotingClassifier
# from xgboost import XGBClassifier

# train_df = pd.read_csv('pre_processing.csv')
# X_train = train_df.drop(columns=['label']).replace([np.inf, -np.inf], np.nan).fillna(0)
# y_train = train_df['label']

# test_df = pd.read_csv('pre_processing2.csv')
# X_test = test_df.replace([np.inf, -np.inf], np.nan).fillna(0)

# rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
# xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# voting_model = VotingClassifier(
#     estimators=[('rf', rf_model), ('xgb', xgb_model)],
#     voting='soft'
# )

# voting_model.fit(X_train, y_train)

# y_pred = voting_model.predict(X_test)

# submission_df = pd.DataFrame({
#     'id': range(1, len(X_test) + 1),
#     'label': y_pred
# })
# submission_df.to_csv('CCC.csv', index=False)
